# Time Series Analysis - Assignment

Welcome to the assignment for Time Series Analysis.

Time series analysis focuses on understanding and modeling data that is collected sequentially over time. Unlike standard machine learning problems where observations are assumed to be independent, time series data contains temporal dependencies, making its analysis and forecasting fundamentally different. Examples of time series data include stock prices, weather measurements, energy consumption, and sales records.

In this assignment, you will explore essential techniques used to analyze and forecast time-dependent data. You will learn how to visualize temporal patterns, identify trends and seasonality, and prepare time series data for modeling. Proper handling of time dependencies is crucial, as incorrect data splitting or assumptions can lead to misleading results.

Time series models aim to capture the underlying structure of sequential data in order to make accurate predictions about future values. Some methods focus on smoothing past observations, while others explicitly model temporal correlations. Understanding these approaches and their assumptions is essential for reliable forecasting and analysis.

**What You Will Do in This Assignment**

* Explore and visualize time series data to identify trends, seasonality, and patterns.
* Decompose time series into trend, seasonal, and residual components.
* Test for stationarity and apply transformations to make a series stationary.
* Analyze temporal dependencies using autocorrelation (ACF) and partial autocorrelation (PACF).
* Implement simple forecasting methods such as moving average and exponential smoothing.
* Split time series data correctly for training and evaluation using temporal validation.
* Build moving average models from scratch, including simple and weighted variants.
* Implement single exponential smoothing from scratch and analyze its behavior.

Let's begin your journey into time series analysis! 

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to Time Series](#1)
2. [Time Series Exploration](#2)
   - [Exercise 1: Load and Visualize Time Series](#ex-1)
3. [Time Series Decomposition](#3)
   - [Exercise 2: Decompose Time Series](#ex-2)
4. [Stationarity Testing](#4)
   - [Exercise 3: Test Stationarity and Transform](#ex-3)
5. [Autocorrelation Analysis](#5)
   - [Exercise 4: Compute and Interpret ACF/PACF](#ex-4)
6. [Simple Forecasting Methods](#6)
   - [Exercise 5: Moving Average and Exponential Smoothing](#ex-5)
7. [Time Series Train/Test Split](#7)
   - [Exercise 6: Implement Temporal Validation](#ex-6)
8. [From-Scratch Implementations](#8)
   - [Exercise 7: Moving Average from Scratch](#ex-7)
   - [Exercise 8: Exponential Smoothing from Scratch](#ex-8)
9. [Conclusion](#9)

<a name='1'></a>
## 1 - Introduction to Time Series

**Time Series** is a sequence of data points collected at successive time intervals. Unlike cross-sectional data, time series data has temporal ordering that must be respected.

### Key Components:

1. **Trend (T)**: Long-term increase or decrease in the data
2. **Seasonality (S)**: Regular patterns that repeat at fixed intervals
3. **Cyclical (C)**: Fluctuations that occur at irregular intervals
4. **Residual/Noise (R)**: Random variation remaining after removing other components

### Decomposition Models:

**Additive Model** (constant seasonal variation):
$$Y_t = T_t + S_t + R_t$$

**Multiplicative Model** (seasonal variation proportional to level):
$$Y_t = T_t \times S_t \times R_t$$

### Stationarity:

A time series is **stationary** if its statistical properties don't change over time:
- Constant mean: $E[Y_t] = \mu$
- Constant variance: $Var[Y_t] = \sigma^2$
- Constant autocovariance: $Cov[Y_t, Y_{t+k}]$ depends only on lag $k$

### Applications:

- **Finance**: Stock prices, trading volumes, risk analysis
- **Economics**: GDP, inflation, unemployment rates
- **Weather**: Temperature, rainfall, wind patterns
- **Healthcare**: Patient vitals, disease spread, hospital admissions
- **Business**: Sales forecasting, inventory management, demand planning

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import unittests
warnings.filterwarnings('ignore')

# Time series specific
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def plot_time_series(data, title="Time Series", xlabel="Time", ylabel="Value"):
    """Plot time series data."""
    plt.figure(figsize=(12, 6))
    plt.plot(data.index, data.values, linewidth=2)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_components(series, trend=None, seasonal=None, residual=None):
    """Plot original series and its components."""
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    
    # Original
    axes[0].plot(series.index, series.values, color='blue', linewidth=2)
    axes[0].set_ylabel('Original', fontsize=11)
    axes[0].set_title('Time Series Decomposition', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Trend
    if trend is not None:
        axes[1].plot(trend.index, trend.values, color='red', linewidth=2)
        axes[1].set_ylabel('Trend', fontsize=11)
        axes[1].grid(True, alpha=0.3)
    
    # Seasonal
    if seasonal is not None:
        axes[2].plot(seasonal.index, seasonal.values, color='green', linewidth=2)
        axes[2].set_ylabel('Seasonal', fontsize=11)
        axes[2].grid(True, alpha=0.3)
    
    # Residual
    if residual is not None:
        axes[3].plot(residual.index, residual.values, color='orange', linewidth=1)
        axes[3].set_ylabel('Residual', fontsize=11)
        axes[3].set_xlabel('Time', fontsize=11)
        axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_forecast(train, test, forecast, title="Forecast vs Actual"):
    """Plot training data, test data, and forecasts."""
    plt.figure(figsize=(12, 6))
    
    # Plot training data
    plt.plot(train.index, train.values, label='Training Data', 
             color='blue', linewidth=2)
    
    # Plot test data
    plt.plot(test.index, test.values, label='Actual', 
             color='green', linewidth=2, marker='o', markersize=4)
    
    # Plot forecast
    plt.plot(forecast.index, forecast.values, label='Forecast', 
             color='red', linewidth=2, linestyle='--', marker='s', markersize=4)
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def calculate_metrics(actual, predicted):
    """Calculate forecasting metrics."""
    mae = np.mean(np.abs(actual - predicted))
    mse = np.mean((actual - predicted) ** 2)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape
    }

print("Helper functions defined!")

<a name='2'></a>
## 2 - Time Series Exploration

The first step in time series analysis is to understand the data through visualization and descriptive statistics.

### What to Look For:

1. **Trend**: Is there a long-term upward or downward movement?
2. **Seasonality**: Are there regular patterns that repeat?
3. **Outliers**: Are there unusual spikes or drops?
4. **Variance**: Does variability change over time (heteroscedasticity)?
5. **Missing Values**: Are there gaps in the time series?

### Visualization Types:

- **Line Plot**: Shows temporal evolution
- **Seasonal Plot**: Compares same periods across years
- **Box Plot by Period**: Shows distribution for each season/month
- **Lag Plot**: Shows relationship with lagged values
- **Rolling Statistics**: Shows moving average and standard deviation

<a name='ex-1'></a>
### Exercise 1 - Load and Visualize Time Series

**Your Task:**

Implement the `explore_time_series` function that:
1. Creates a synthetic time series with trend and seasonality
2. Computes descriptive statistics
3. Visualizes the series with rolling statistics
4. Creates seasonal subseries plots

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For creating time series:**
* Date range: `pd.date_range(start='2020-01-01', periods=n_periods, freq='D')`
* Trend: `np.linspace(start, end, n_periods)`
* Seasonality: `amplitude * np.sin(2 * np.pi * np.arange(n_periods) / period)`
* Combine: `trend + seasonality + noise`

**For rolling statistics:**
* Rolling mean: `series.rolling(window=window_size).mean()`
* Rolling std: `series.rolling(window=window_size).std()`

**For statistics:**
* Basic stats: `series.describe()`
* Additional: `series.mean()`, `series.std()`, `series.min()`, `series.max()`

</details>

In [ ]:
# GRADED FUNCTION: create_synthetic_series
def create_synthetic_series(n_periods=365, trend_strength=0.5, 
                           seasonal_period=30, seasonal_strength=10,
                           noise_level=2, start_date='2020-01-01'):
    """
    Create synthetic time series with trend, seasonality, and noise.
    
    Args:
        n_periods: Number of time periods
        trend_strength: Slope of linear trend
        seasonal_period: Period of seasonality (e.g., 30 for monthly)
        seasonal_strength: Amplitude of seasonal component
        noise_level: Standard deviation of random noise
        start_date: Starting date for time series
    
    Returns:
        pd.Series with DatetimeIndex
    """
    ### START CODE HERE ### (≈ 8 lines)
    # Create date range
    dates = None
    
    # Create trend component
    trend = None
    
    # Create seasonal component
    seasonal = None
    
    # Create noise component
    noise = None
    
    # Combine components
    values = None
    
    # Create Series
    series = None
    ### END CODE HERE ###
    
    return series


In [ ]:
unittests.exercise_1(create_synthetic_series)

#### `explore_time_series` - Explore with Rolling Statistics

Compute rolling mean and standard deviation over a sliding window and return basic descriptive statistics of the series.

In [ ]:
# GRADED FUNCTION: explore_time_series

def explore_time_series(series, window=30):
    """
    Explore time series data with visualizations and statistics.
    
    Args:
        series: Time series data (pd.Series)
        window: Window size for rolling statistics
    
    Returns:
        dict with statistics
    """
    ### START CODE HERE ### (≈ 6 lines)
    # Compute rolling mean
    rolling_mean = None
    
    # Compute rolling standard deviation
    rolling_std = None
    
    # Compute statistics
    stats = {
        'mean': None,
        'std': None,
        'min': None,
        'max': None,
        'count': None
    }
    ### END CODE HERE ###
    
    # Plot original series with rolling statistics
    plt.figure(figsize=(12, 6))
    plt.plot(series.index, series.values, label='Original', color='blue', alpha=0.6, linewidth=1.5)
    plt.plot(rolling_mean.index, rolling_mean.values, label=f'Rolling Mean (window={window})', color='red', linewidth=2)
    plt.plot(rolling_std.index, rolling_std.values, label=f'Rolling Std (window={window})', color='green', linewidth=2)
    plt.title('Time Series with Rolling Statistics', fontsize=14, fontweight='bold')
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return stats

In [ ]:
unittests.exercise_2(explore_time_series)

In [ ]:
# Create synthetic time series
ts_data = create_synthetic_series(n_periods=365, trend_strength=0.5,
                                  seasonal_period=30, seasonal_strength=10,
                                  noise_level=2)

print("Time Series Created:")
print(f"Shape: {ts_data.shape}")
print(f"Date range: {ts_data.index[0]} to {ts_data.index[-1]}")
print(f"\nFirst 5 values:\n{ts_data.head()}")

# Explore the time series
stats = explore_time_series(ts_data, window=30)

print("\nDescriptive Statistics:")
print(f"Mean: {stats['mean']:.2f}")
print(f"Std Dev: {stats['std']:.2f}")
print(f"Min: {stats['min']:.2f}")
print(f"Max: {stats['max']:.2f}")
print(f"Count: {stats['count']}")

##### **Expected Output**
```
Time Series Created:
Shape: (365,)
Date range: 2020-01-01 to 2020-12-30

First 5 values:
2020-01-01    XX.XX
2020-01-02    XX.XX
...

Descriptive Statistics:
Mean: XX.XX
Std Dev: XX.XX
Min: XX.XX
Max: XX.XX
Count: 365
```
You should see an upward trend with clear seasonality and the rolling mean smoothing the noise.

<a name='3'></a>
## 3 - Time Series Decomposition

**Decomposition** separates a time series into its constituent components to better understand the underlying patterns.

### Decomposition Methods:

1. **Classical Decomposition**:
   - Uses moving averages to estimate trend
   - Averages detrended values to estimate seasonality
   - Simple but assumes constant seasonal pattern

2. **STL Decomposition** (Seasonal and Trend decomposition using Loess):
   - More flexible and robust
   - Handles changing seasonal patterns
   - Less sensitive to outliers

### Additive vs Multiplicative:

**Additive**: $Y_t = T_t + S_t + R_t$
- Use when seasonal variation is constant
- Appropriate when series doesn't change much in amplitude

**Multiplicative**: $Y_t = T_t \times S_t \times R_t$
- Use when seasonal variation grows with trend
- Can convert to additive using log: $\log(Y_t) = \log(T_t) + \log(S_t) + \log(R_t)$

### Applications:
- Identify and remove seasonality for better modeling
- Detect anomalies in residuals
- Forecast trend and seasonality separately

<a name='ex-2'></a>
### Exercise 2 - Decompose Time Series

**Your Task:**

Implement the `decompose_series` function that:
1. Applies seasonal decomposition (additive or multiplicative)
2. Extracts trend, seasonal, and residual components
3. Visualizes all components
4. Computes strength of trend and seasonality

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For decomposition:**
* Use seasonal_decompose: `result = seasonal_decompose(series, model='additive', period=period)`
* Extract components: `result.trend`, `result.seasonal`, `result.resid`

**For strength metrics:**
* Variance of component: `np.var(component.dropna())`
* Strength of trend: `1 - var(residual) / var(detrended)`
* Strength of seasonality: `1 - var(residual) / var(deseasonalized)`

**Note**: Use `.dropna()` to handle NaN values from decomposition

</details>

In [ ]:
# GRADED FUNCTION: decompose_series
def decompose_series(series, model='additive', period=30):
    """
    Decompose time series into trend, seasonal, and residual components.
    
    Args:
        series: Time series data (pd.Series)
        model: 'additive' or 'multiplicative'
        period: Seasonal period
    
    Returns:
        dict with 'trend', 'seasonal', 'residual', 'strength_trend', 'strength_seasonal'
    """
    ### START CODE HERE ### (≈ 12 lines)
    # Apply decomposition
    decomposition = None
    
    # Extract components
    trend = None
    seasonal = None
    residual = None
    
    # Compute strength of trend
    # Detrended = Original - Trend
    detrended = (series - trend).dropna()
    resid_clean = residual.dropna()
    strength_trend = None  # 1 - var(residual) / var(detrended)
    
    # Compute strength of seasonality
    # Deseasonalized = Original - Seasonal
    deseasonalized = (series - seasonal).dropna()
    strength_seasonal = None  # 1 - var(residual) / var(deseasonalized)
    ### END CODE HERE ###
    
    # Visualize
    plot_components(series, trend, seasonal, residual)
    
    return {
        'trend': trend,
        'seasonal': seasonal,
        'residual': residual,
        'strength_trend': strength_trend,
        'strength_seasonal': strength_seasonal
    }

In [ ]:
unittests.exercise_3(decompose_series)

In [ ]:
# Decompose the time series
decomp_result = decompose_series(ts_data, model='additive', period=30)

print("\nDecomposition Results:")
print(f"Strength of Trend: {decomp_result['strength_trend']:.4f}")
print(f"Strength of Seasonality: {decomp_result['strength_seasonal']:.4f}")

# Analyze residuals
residuals = decomp_result['residual'].dropna()
print(f"\nResidual Statistics:")
print(f"Mean: {residuals.mean():.4f} (should be close to 0)")
print(f"Std Dev: {residuals.std():.4f}")
print(f"Min: {residuals.min():.4f}")
print(f"Max: {residuals.max():.4f}")

# Plot histogram of residuals
plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(residuals.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean = {residuals.mean():.2f}')
plt.title('Distribution of Residuals', fontsize=14, fontweight='bold')
plt.xlabel('Residual Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

##### **Expected Output**
```
Decomposition Results:
Strength of Trend: 0.9XXX
Strength of Seasonality: 0.9XXX

Residual Statistics:
Mean: 0.XXXX (should be close to 0)
Std Dev: X.XXXX
Min: -X.XXXX
Max: X.XXXX
```
Strong trend and seasonality should have values close to 1. Residuals should be normally distributed around 0.

<a name='4'></a>
## 4 - Stationarity Testing

**Stationarity** is a crucial concept in time series analysis. Many forecasting methods assume or require stationary data.

### Why Stationarity Matters:

- Makes time series easier to model
- Statistical properties are consistent over time
- Improves forecast reliability
- Required for ARIMA models

### Augmented Dickey-Fuller (ADF) Test:

Tests the null hypothesis: "Series has a unit root (non-stationary)"

**Test Statistic**: More negative = more likely stationary

**Decision Rule**:
- If p-value < 0.05: Reject null → **Stationary**
- If p-value ≥ 0.05: Fail to reject → **Non-stationary**

### Making Series Stationary:

1. **Differencing**: $Y'_t = Y_t - Y_{t-1}$
   - Removes trend
   - Can apply multiple times
   - First-order: $d=1$, Second-order: $d=2$

2. **Log Transform**: $Y'_t = \log(Y_t)$
   - Stabilizes variance
   - Converts multiplicative to additive

3. **Detrending**: $Y'_t = Y_t - T_t$
   - Removes trend component
   - Linear or polynomial detrending

4. **Seasonal Differencing**: $Y'_t = Y_t - Y_{t-s}$
   - Removes seasonality
   - $s$ = seasonal period

<a name='ex-3'></a>
### Exercise 3 - Test Stationarity and Transform

**Your Task:**

Implement two functions:
1. `test_stationarity`: Apply ADF test and return results
2. `make_stationary`: Apply differencing to make series stationary

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For ADF test:**
* Apply test: `result = adfuller(series.dropna(), autolag='AIC')`
* Test statistic: `result[0]`
* P-value: `result[1]`
* Critical values: `result[4]` (dictionary)
* Interpretation: `p_value < 0.05` means stationary

**For differencing:**
* First difference: `series.diff(periods=1).dropna()`
* Second difference: Apply diff twice
* Seasonal difference: `series.diff(periods=seasonal_period).dropna()`

</details>

In [ ]:
# GRADED FUNCTION: test_stationarity
def test_stationarity(series):
    """
    Test if time series is stationary using Augmented Dickey-Fuller test.
    
    Args:
        series: Time series data (pd.Series)
    
    Returns:
        dict with 'adf_stat', 'p_value', 'critical_values', 'is_stationary'
    """
    ### START CODE HERE ### (≈ 6 lines)
    # Apply ADF test
    adf_result = None
    
    # Extract values
    adf_stat = None
    p_value = None
    critical_values = None
    
    # Determine if stationary
    is_stationary = None  # p_value < 0.05
    ### END CODE HERE ###
    
    return {
        'adf_stat': adf_stat,
        'p_value': p_value,
        'critical_values': critical_values,
        'is_stationary': is_stationary
    }


In [ ]:
unittests.exercise_4(test_stationarity)

#### `make_stationary` - Apply Differencing or Log Transform

Implement `make_stationary` that supports `method='diff'` (differencing of order `order`) and `method='log'` (log transform followed by differencing). Also support optional `seasonal_period` differencing.

In [ ]:
# GRADED FUNCTION: make_stationary
# GRADED FUNCTION: make_stationary
def make_stationary(series, method='diff', order=1, seasonal_period=None):
    """
    Transform series to make it stationary.
    
    Args:
        series: Time series data (pd.Series)
        method: 'diff' for differencing, 'log' for log transform
        order: Order of differencing (1 or 2)
        seasonal_period: Period for seasonal differencing (optional)
    
    Returns:
        Transformed series (pd.Series)
    """
    transformed = series.copy()
    
    ### START CODE HERE ### (≈ 10 lines)
    if method == 'log':
        # Log transform
        transformed = None
    
    if method == 'diff' or method == 'log':
        # Apply differencing
        for _ in range(order):
            transformed = None
        
        # Apply seasonal differencing if specified
        if seasonal_period is not None:
            transformed = None
    ### END CODE HERE ###
    
    return transformed

In [ ]:
unittests.exercise_5(make_stationary)

In [ ]:
# Test stationarity of original series
print("=" * 60)
print("STATIONARITY TEST - ORIGINAL SERIES")
print("=" * 60)

original_test = test_stationarity(ts_data)
print(f"\nADF Statistic: {original_test['adf_stat']:.4f}")
print(f"P-value: {original_test['p_value']:.4f}")
print(f"Critical Values:")
for key, value in original_test['critical_values'].items():
    print(f"  {key}: {value:.4f}")
print(f"\nIs Stationary: {original_test['is_stationary']}")

# Make series stationary using first-order differencing
ts_diff = make_stationary(ts_data, method='diff', order=1)

print("\n" + "=" * 60)
print("STATIONARITY TEST - AFTER DIFFERENCING")
print("=" * 60)

diff_test = test_stationarity(ts_diff)
print(f"\nADF Statistic: {diff_test['adf_stat']:.4f}")
print(f"P-value: {diff_test['p_value']:.4f}")
print(f"Critical Values:")
for key, value in diff_test['critical_values'].items():
    print(f"  {key}: {value:.4f}")
print(f"\nIs Stationary: {diff_test['is_stationary']}")

# Visualize original vs differenced
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Original series
ax1.plot(ts_data.index, ts_data.values, linewidth=2, color='blue')
ax1.set_title('Original Time Series (Non-Stationary)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Value', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=ts_data.mean(), color='red', linestyle='--', 
            label=f'Mean = {ts_data.mean():.2f}')
ax1.legend()

# Differenced series
ax2.plot(ts_diff.index, ts_diff.values, linewidth=2, color='green')
ax2.set_title('Differenced Time Series (Stationary)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time', fontsize=11)
ax2.set_ylabel('Value', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=ts_diff.mean(), color='red', linestyle='--', 
            label=f'Mean = {ts_diff.mean():.2f}')
ax2.legend()

plt.tight_layout()
plt.show()

##### **Expected Output**
```
============================================================
STATIONARITY TEST - ORIGINAL SERIES
============================================================

ADF Statistic: -X.XXXX
P-value: 0.XXXX (likely > 0.05)
Critical Values:
  1%: -X.XXXX
  5%: -X.XXXX
  10%: -X.XXXX

Is Stationary: False

============================================================
STATIONARITY TEST - AFTER DIFFERENCING
============================================================

ADF Statistic: -XX.XXXX (much more negative)
P-value: 0.0000 (< 0.05)
Critical Values:
  1%: -X.XXXX
  5%: -X.XXXX
  10%: -X.XXXX

Is Stationary: True
```
Original series should be non-stationary. After differencing, it should become stationary.

<a name='5'></a>
## 5 - Autocorrelation Analysis

**Autocorrelation** measures the correlation between a time series and lagged versions of itself.

### Autocorrelation Function (ACF):

Correlation between $Y_t$ and $Y_{t-k}$ at lag $k$:
$$\rho_k = \frac{Cov(Y_t, Y_{t-k})}{\sigma^2}$$

**Properties**:
- $\rho_0 = 1$ (perfect correlation with itself)
- $-1 \leq \rho_k \leq 1$
- Decays slowly for non-stationary series
- Shows direct and indirect effects

### Partial Autocorrelation Function (PACF):

Correlation between $Y_t$ and $Y_{t-k}$ after removing effects of intermediate lags:

**Properties**:
- Shows only direct relationship
- Removes indirect correlations
- Helps identify AR order

### Model Identification:

| Model | ACF Pattern | PACF Pattern |
|-------|-------------|-------------|
| **AR(p)** | Decays gradually | Cuts off after lag p |
| **MA(q)** | Cuts off after lag q | Decays gradually |
| **ARMA(p,q)** | Decays gradually | Decays gradually |

### Confidence Intervals:

95% confidence bounds: $\pm \frac{1.96}{\sqrt{n}}$

Values outside these bounds are significant.

<a name='ex-4'></a>
### Exercise 4 - Compute and Interpret ACF/PACF

**Your Task:**

Implement the `analyze_autocorrelation` function that:
1. Computes ACF and PACF values
2. Plots ACF and PACF with confidence intervals
3. Identifies significant lags
4. Suggests model order based on patterns

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For ACF/PACF computation:**
* ACF values: `acf_values = acf(series.dropna(), nlags=nlags)`
* PACF values: `pacf_values = pacf(series.dropna(), nlags=nlags)`

**For plotting:**
* Use plot_acf: `plot_acf(series.dropna(), lags=nlags, ax=ax)`
* Use plot_pacf: `plot_pacf(series.dropna(), lags=nlags, ax=ax)`

**For significance:**
* Confidence bound: `1.96 / np.sqrt(len(series))`
* Significant lags: Where abs(value) > confidence bound

**For interpretation:**
* Count significant lags in PACF → suggests AR order
* Count significant lags in ACF → suggests MA order

</details>

In [ ]:
# GRADED FUNCTION: analyze_autocorrelation
def analyze_autocorrelation(series, nlags=40):
    """
    Analyze autocorrelation structure of time series.
    
    Args:
        series: Time series data (pd.Series)
        nlags: Number of lags to compute
    
    Returns:
        dict with 'acf_values', 'pacf_values', 'significant_acf_lags', 'significant_pacf_lags'
    """
    series_clean = series.dropna()
    
    ### START CODE HERE ### (≈ 10 lines)
    # Compute ACF
    acf_values = None
    
    # Compute PACF
    pacf_values = None
    
    # Compute confidence bound for significance
    confidence_bound = None  # 1.96 / sqrt(n)
    
    # Find significant lags (excluding lag 0)
    significant_acf_lags = None  # [i for i in range(1, len(acf_values)) if abs(acf_values[i]) > confidence_bound]
    significant_pacf_lags = None  # similar for PACF
    ### END CODE HERE ###
    
    # Plot ACF and PACF
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # ACF plot
    plot_acf(series_clean, lags=nlags, ax=ax1)
    ax1.set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Lag', fontsize=11)
    ax1.set_ylabel('Correlation', fontsize=11)
    
    # PACF plot
    plot_pacf(series_clean, lags=nlags, ax=ax2)
    ax2.set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Lag', fontsize=11)
    ax2.set_ylabel('Correlation', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'acf_values': acf_values,
        'pacf_values': pacf_values,
        'significant_acf_lags': significant_acf_lags,
        'significant_pacf_lags': significant_pacf_lags,
        'confidence_bound': confidence_bound
    }

In [ ]:
unittests.exercise_6(analyze_autocorrelation)

In [ ]:
# Analyze autocorrelation of differenced series
autocorr_result = analyze_autocorrelation(ts_diff, nlags=40)

print("Autocorrelation Analysis:")
print(f"\nConfidence Bound: ±{autocorr_result['confidence_bound']:.4f}")
print(f"\nSignificant ACF lags: {autocorr_result['significant_acf_lags'][:10]}...")
print(f"Total significant ACF lags: {len(autocorr_result['significant_acf_lags'])}")
print(f"\nSignificant PACF lags: {autocorr_result['significant_pacf_lags'][:10]}...")
print(f"Total significant PACF lags: {len(autocorr_result['significant_pacf_lags'])}")

# Suggest model order
print("\n" + "="*60)
print("MODEL SUGGESTION")
print("="*60)

# Count early significant lags (first 5)
early_pacf = sum(1 for lag in autocorr_result['significant_pacf_lags'] if lag <= 5)
early_acf = sum(1 for lag in autocorr_result['significant_acf_lags'] if lag <= 5)

print(f"\nSignificant lags in first 5:")
print(f"  PACF: {early_pacf} → Suggests AR({early_pacf})")
print(f"  ACF: {early_acf} → Suggests MA({early_acf})")
print(f"\nRecommended: Consider ARMA({early_pacf},{early_acf}) or simpler variants")

##### **Expected Output**
```
Autocorrelation Analysis:

Confidence Bound: ±0.XXXX

Significant ACF lags: [X, X, X, ...]...
Total significant ACF lags: XX

Significant PACF lags: [X, X, X, ...]...
Total significant PACF lags: XX

============================================================
MODEL SUGGESTION
============================================================

Significant lags in first 5:
  PACF: X → Suggests AR(X)
  ACF: X → Suggests MA(X)

Recommended: Consider ARMA(X,X) or simpler variants
```
You should see clear patterns in ACF/PACF plots with some lags exceeding confidence bounds.

<a name='6'></a>
## 6 - Simple Forecasting Methods

Before diving into complex models, simple forecasting methods often provide good baselines.

### 1. Moving Average (MA):

Average of previous $k$ observations:
$$\hat{Y}_{t+1} = \frac{1}{k} \sum_{i=0}^{k-1} Y_{t-i}$$

**Properties**:
- Smooths short-term fluctuations
- Lags behind trend
- Equal weights for all observations in window

### 2. Weighted Moving Average (WMA):

Weighted average with more recent values getting higher weights:
$$\hat{Y}_{t+1} = \sum_{i=0}^{k-1} w_i \cdot Y_{t-i}$$

where $\sum w_i = 1$ and typically $w_0 > w_1 > ... > w_{k-1}$

### 3. Exponential Smoothing (Simple):

Weighted average with exponentially decaying weights:
$$\hat{Y}_{t+1} = \alpha Y_t + (1-\alpha) \hat{Y}_t$$

Or equivalently:
$$\hat{Y}_{t+1} = \alpha Y_t + \alpha(1-\alpha) Y_{t-1} + \alpha(1-\alpha)^2 Y_{t-2} + ...$$

**Parameters**:
- $\alpha \in [0, 1]$: Smoothing parameter
- High $\alpha$ (e.g., 0.8): Responsive to recent changes
- Low $\alpha$ (e.g., 0.2): More smoothing, less responsive

**Advantages**:
Simple to implement
Computationally efficient
Only needs to store previous forecast
Adapts to changes in data

### 4. Double Exponential Smoothing (Holt's):

Extends simple smoothing to handle trends:
- **Level**: $L_t = \alpha Y_t + (1-\alpha)(L_{t-1} + T_{t-1})$
- **Trend**: $T_t = \beta(L_t - L_{t-1}) + (1-\beta)T_{t-1}$
- **Forecast**: $\hat{Y}_{t+h} = L_t + h \cdot T_t$

<a name='ex-5'></a>
### Exercise 5 - Moving Average and Exponential Smoothing

**Your Task:**

Implement forecasting using:
1. `forecast_moving_average`: Simple moving average
2. `forecast_exponential_smoothing`: Single exponential smoothing using statsmodels

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For Moving Average:**
* Rolling mean: `train.rolling(window=window).mean()`
* Last value as forecast: `ma_values.iloc[-1]`
* Repeat for all forecast periods

**For Exponential Smoothing:**
* Create model: `model = SimpleExpSmoothing(train)`
* Fit: `fit = model.fit(smoothing_level=alpha, optimized=False)`
* Forecast: `forecast = fit.forecast(steps=len(test))`

**For metrics:**
* Use helper function: `calculate_metrics(test.values, forecast.values)`

</details>

In [ ]:
# GRADED FUNCTION: forecast_moving_average
def forecast_moving_average(train, test, window=7):
    """
    Forecast using simple moving average.
    
    Args:
        train: Training time series
        test: Test time series
        window: Window size for moving average
    
    Returns:
        forecast: Predicted values (pd.Series)
    """
    ### START CODE HERE ### (≈ 5 lines)
    # Compute moving average on training data
    ma_values = None
    
    # Use last MA value as forecast for all test periods
    forecast_value = None
    
    # Create forecast series
    forecast = pd.Series([forecast_value] * len(test), index=test.index)
    ### END CODE HERE ###
    
    return forecast


In [ ]:
unittests.exercise_7(forecast_moving_average)

#### `forecast_exponential_smoothing` - Statsmodels Exponential Smoothing

Use `SimpleExpSmoothing` from statsmodels to fit a model on `train` and forecast `len(test)` steps.

In [ ]:
# GRADED FUNCTION: forecast_exponential_smoothing
# GRADED FUNCTION: forecast_exponential_smoothing
def forecast_exponential_smoothing(train, test, alpha=0.3):
    """
    Forecast using single exponential smoothing.
    
    Args:
        train: Training time series
        test: Test time series
        alpha: Smoothing parameter (0 to 1)
    
    Returns:
        forecast: Predicted values (pd.Series)
    """
    ### START CODE HERE ### (≈ 5 lines)
    # Create and fit model
    model = None
    fit = None
    
    # Generate forecast
    forecast = None
    ### END CODE HERE ###
    
    return forecast

In [ ]:
unittests.exercise_8(forecast_exponential_smoothing)

In [ ]:
# Split data into train and test
train_size = int(len(ts_data) * 0.8)
train = ts_data[:train_size]
test = ts_data[train_size:]

print(f"Train size: {len(train)}")
print(f"Test size: {len(test)}")

# Moving Average forecast
ma_forecast = forecast_moving_average(train, test, window=30)
ma_metrics = calculate_metrics(test.values, ma_forecast.values)

print("\n" + "="*60)
print("MOVING AVERAGE (window=30)")
print("="*60)
for metric, value in ma_metrics.items():
    print(f"{metric}: {value:.4f}")

# Exponential Smoothing forecast
es_forecast = forecast_exponential_smoothing(train, test, alpha=0.3)
es_metrics = calculate_metrics(test.values, es_forecast.values)

print("\n" + "="*60)
print("EXPONENTIAL SMOOTHING (alpha=0.3)")
print("="*60)
for metric, value in es_metrics.items():
    print(f"{metric}: {value:.4f}")

# Visualize forecasts
plt.figure(figsize=(14, 7))
plt.plot(train.index, train.values, label='Training', color='blue', linewidth=2)
plt.plot(test.index, test.values, label='Actual', color='green', 
         linewidth=2, marker='o', markersize=3)
plt.plot(ma_forecast.index, ma_forecast.values, label='MA Forecast', 
         color='red', linewidth=2, linestyle='--')
plt.plot(es_forecast.index, es_forecast.values, label='ES Forecast', 
         color='orange', linewidth=2, linestyle='-.')
plt.title('Simple Forecasting Methods Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

##### **Expected Output**
```
Train size: 292
Test size: 73

============================================================
MOVING AVERAGE (window=30)
============================================================
MAE: XX.XXXX
MSE: XXX.XXXX
RMSE: XX.XXXX
MAPE: XX.XXXX

============================================================
EXPONENTIAL SMOOTHING (alpha=0.3)
============================================================
MAE: XX.XXXX
MSE: XXX.XXXX
RMSE: XX.XXXX
MAPE: XX.XXXX
```
Exponential smoothing should perform better than simple moving average on trending data.

<a name='7'></a>
## 7 - Time Series Train/Test Split

**CRITICAL**: Time series data requires special handling for train/test splitting!

### Wrong Approach: Random Split

```python
# DON'T DO THIS!
X_train, X_test = train_test_split(data, test_size=0.2, shuffle=True)
```

**Why wrong**: Violates temporal ordering, causes data leakage

### Correct Approach: Temporal Split

Always split chronologically to respect time order:

```
[----------Training Data----------][--Test Data--]
        Past                        Future
```

### Split Methods:

1. **Simple Holdout**:
   - Single train/test split
   - Train on past, test on recent
   - Most common

2. **Rolling Window**:
   - Fixed-size training window
   - Moves forward in time
   - Useful when recent data is most relevant

3. **Expanding Window**:
   - Growing training set
   - Test on next period
   - More data = better for long-term

4. **Time Series Cross-Validation**:
   ```
   Fold 1: [train] [test]
   Fold 2: [train---] [test]
   Fold 3: [train------] [test]
   ```

### Best Practices:

Use 70-80% for training, 20-30% for testing
Consider gap between train and test (forecasting horizon)
Validate on multiple periods if possible
Be aware of seasonal patterns
Never shuffle time series data
Don't use future data to predict past

<a name='ex-6'></a>
### Exercise 6 - Implement Temporal Validation

**Your Task:**

Implement functions for proper time series validation:
1. `temporal_train_test_split`: Simple temporal split
2. `time_series_cv`: Rolling window cross-validation

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For simple split:**
* Calculate split point: `split_idx = int(len(series) * train_ratio)`
* Train: `series[:split_idx]`
* Test: `series[split_idx:]`

**For cross-validation:**
* Initial train size: `initial_train_size`
* Step size: `step`
* Loop: Create splits moving forward in time
* Each fold: `train = series[start:end]`, `test = series[end:end+test_size]`

**Tips:**
* Ensure test set doesn't go beyond series length
* Store (train_indices, test_indices) tuples

</details>

In [ ]:
# GRADED FUNCTION: temporal_train_test_split
def temporal_train_test_split(series, train_ratio=0.8):
    """
    Split time series into training and test sets temporally.
    
    Args:
        series: Time series data (pd.Series)
        train_ratio: Proportion for training (0 to 1)
    
    Returns:
        train, test: Training and test series
    """
    ### START CODE HERE ### (≈ 4 lines)
    # Calculate split point
    split_idx = None
    
    # Split chronologically
    train = None
    test = None
    ### END CODE HERE ###
    
    return train, test


In [ ]:
unittests.exercise_9(temporal_train_test_split)

#### `time_series_cv` - Rolling Window Cross-Validation

Implement `time_series_cv` that creates an expanding-window cross-validation: each fold increases the training set by `test_size`, keeping the test window fixed.

In [ ]:
# GRADED FUNCTION: time_series_cv
# GRADED FUNCTION: time_series_cv
def time_series_cv(series, n_splits=5, test_size=30):
    """
    Time series cross-validation with expanding window.
    
    Args:
        series: Time series data (pd.Series)
        n_splits: Number of CV folds
        test_size: Size of test set in each fold
    
    Returns:
        List of (train, test) tuples
    """
    splits = []
    n = len(series)
    
    ### START CODE HERE ### (≈ 10 lines)
    # Calculate initial training size
    initial_train_size = None  # n - (n_splits * test_size)
    
    # Create expanding window splits
    for i in range(n_splits):
        # Calculate train end index
        train_end = None
        
        # Calculate test end index
        test_end = None
        
        # Ensure we don't exceed series length
        if test_end > n:
            break
        
        # Create train and test sets
        train = None
        test = None
        
        splits.append((train, test))
    ### END CODE HERE ###
    
    return splits

In [ ]:
unittests.exercise_10(time_series_cv)

In [ ]:
# Test simple temporal split
train, test = temporal_train_test_split(ts_data, train_ratio=0.8)

print("Simple Temporal Split:")
print(f"Train: {train.index[0]} to {train.index[-1]} ({len(train)} samples)")
print(f"Test: {test.index[0]} to {test.index[-1]} ({len(test)} samples)")
print(f"\nNo overlap: {train.index[-1] < test.index[0]}")

# Visualize split
plt.figure(figsize=(14, 6))
plt.plot(train.index, train.values, label='Training Set', color='blue', linewidth=2)
plt.plot(test.index, test.values, label='Test Set', color='red', linewidth=2)
plt.axvline(x=train.index[-1], color='green', linestyle='--', 
            linewidth=2, label='Split Point')
plt.title('Temporal Train/Test Split', fontsize=14, fontweight='bold')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Test time series cross-validation
cv_splits = time_series_cv(ts_data, n_splits=5, test_size=20)

print("\nTime Series Cross-Validation:")
print(f"Number of folds: {len(cv_splits)}\n")

for i, (train_fold, test_fold) in enumerate(cv_splits, 1):
    print(f"Fold {i}:")
    print(f"  Train: {train_fold.index[0]} to {train_fold.index[-1]} ({len(train_fold)} samples)")
    print(f"  Test:  {test_fold.index[0]} to {test_fold.index[-1]} ({len(test_fold)} samples)")

# Visualize CV folds
fig, axes = plt.subplots(len(cv_splits), 1, figsize=(14, 10))

for i, (train_fold, test_fold) in enumerate(cv_splits):
    ax = axes[i]
    ax.plot(train_fold.index, train_fold.values, color='blue', linewidth=1.5, label='Train')
    ax.plot(test_fold.index, test_fold.values, color='red', linewidth=1.5, label='Test')
    ax.set_ylabel(f'Fold {i+1}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='upper left')
        ax.set_title('Time Series Cross-Validation (Expanding Window)', 
                    fontsize=12, fontweight='bold')
    if i == len(cv_splits) - 1:
        ax.set_xlabel('Time', fontsize=11)

plt.tight_layout()
plt.show()

##### **Expected Output**
```
Simple Temporal Split:
Train: 2020-01-01 to 2020-10-XX (292 samples)
Test: 2020-10-XX to 2020-12-30 (73 samples)

No overlap: True

Time Series Cross-Validation:
Number of folds: 5

Fold 1:
  Train: 2020-01-01 to 2020-XX-XX (XXX samples)
  Test:  2020-XX-XX to 2020-XX-XX (20 samples)
...
```
Each fold should have expanding training set and fixed-size test set moving forward.

<a name='8'></a>
## 8 - From-Scratch Implementations

Understanding how forecasting algorithms work internally helps you customize and optimize them for specific use cases.

<a name='ex-7'></a>
### Exercise 7 - Moving Average from Scratch

**Your Task:**

Implement two variants:
1. `simple_moving_average_scratch`: Equal weights
2. `weighted_moving_average_scratch`: Linear weights (more recent = higher weight)

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For Simple MA:**
* Loop from window to end of series
* For each position: `ma[i] = np.mean(series[i-window:i])`
* First (window-1) values will be NaN

**For Weighted MA:**
* Create weights: `weights = np.arange(1, window+1)` (linear)
* Normalize: `weights = weights / weights.sum()`
* Compute: `wma[i] = np.dot(series[i-window:i], weights)`

**Alternative weights:**
* Exponential: `np.exp(np.arange(window))`
* Custom: Any sequence summing to 1

</details>

In [ ]:
# GRADED FUNCTION: simple_moving_average_scratch
def simple_moving_average_scratch(series, window=7):
    """
    Compute simple moving average from scratch.
    
    Args:
        series: Time series data (array-like)
        window: Window size
    
    Returns:
        Moving averages (numpy array, same length as input)
    """
    series = np.array(series)
    n = len(series)
    ma = np.full(n, np.nan)  # Initialize with NaN
    
    ### START CODE HERE ### (≈ 4 lines)
    # Compute moving average for each position
    for i in range(window, n + 1):
        # Average of previous 'window' values
        ma[i-1] = None
    ### END CODE HERE ###
    
    return ma


In [ ]:
unittests.exercise_11(simple_moving_average_scratch)

#### `weighted_moving_average_scratch` - Linear Weighted Variant

Compute a weighted moving average where weights increase linearly from 1 to `window` (most recent observation gets the highest weight). Normalize weights to sum to 1.

In [ ]:
# GRADED FUNCTION: weighted_moving_average_scratch
# GRADED FUNCTION: weighted_moving_average_scratch
def weighted_moving_average_scratch(series, window=7):
    """
    Compute weighted moving average from scratch.
    More recent values have higher weights.
    
    Args:
        series: Time series data (array-like)
        window: Window size
    
    Returns:
        Weighted moving averages (numpy array, same length as input)
    """
    series = np.array(series)
    n = len(series)
    wma = np.full(n, np.nan)  # Initialize with NaN
    
    ### START CODE HERE ### (≈ 7 lines)
    # Create linear weights (1, 2, 3, ..., window)
    weights = None
    
    # Normalize weights to sum to 1
    weights = None
    
    # Compute weighted moving average for each position
    for i in range(window, n + 1):
        # Weighted average of previous 'window' values
        wma[i-1] = None
    ### END CODE HERE ###
    
    return wma

In [ ]:
unittests.exercise_12(weighted_moving_average_scratch)

In [ ]:
# Test moving averages from scratch
window = 7

# Compute using scratch implementations
sma_scratch = simple_moving_average_scratch(ts_data.values, window=window)
wma_scratch = weighted_moving_average_scratch(ts_data.values, window=window)

# Compare with pandas rolling mean
sma_pandas = ts_data.rolling(window=window).mean()

# Verify correctness
print("Verification (Simple MA):")
print(f"Max difference from pandas: {np.nanmax(np.abs(sma_scratch - sma_pandas.values)):.10f}")
print(f"Should be very close to 0.0")

# Visualize
plt.figure(figsize=(14, 7))
plt.plot(ts_data.index, ts_data.values, label='Original', 
         color='blue', alpha=0.4, linewidth=1)
plt.plot(ts_data.index, sma_scratch, label=f'Simple MA (window={window})', 
         color='red', linewidth=2)
plt.plot(ts_data.index, wma_scratch, label=f'Weighted MA (window={window})', 
         color='green', linewidth=2)
plt.title('Moving Averages from Scratch', fontsize=14, fontweight='bold')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show first few values
print("\nFirst 10 values comparison:")
comparison_df = pd.DataFrame({
    'Original': ts_data.values[:10],
    'Simple MA': sma_scratch[:10],
    'Weighted MA': wma_scratch[:10]
})
print(comparison_df.to_string())

##### **Expected Output**
```
Verification (Simple MA):
Max difference from pandas: 0.0000000000 (or very small)
Should be very close to 0.0

First 10 values comparison:
   Original  Simple MA  Weighted MA
0     XX.XX        NaN          NaN
1     XX.XX        NaN          NaN
...
6     XX.XX        NaN          NaN
7     XX.XX     XX.XX        XX.XX
8     XX.XX     XX.XX        XX.XX
9     XX.XX     XX.XX        XX.XX
```
Weighted MA should be more responsive (closer to recent values) than simple MA.

<a name='ex-8'></a>
### Exercise 8 - Exponential Smoothing from Scratch

**Your Task:**

Implement single exponential smoothing from scratch using the recursive formula:
$$S_t = \alpha Y_t + (1-\alpha) S_{t-1}$$

where $S_0 = Y_0$ (first observation)

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For initialization:**
* First smoothed value: `smoothed[0] = series[0]`

**For recursion:**
* Loop from 1 to n
* Update: `smoothed[t] = alpha * series[t] + (1 - alpha) * smoothed[t-1]`

**For forecasting:**
* One-step ahead: Use last smoothed value
* Multiple steps: Repeat same value (level forecast)

**Alpha interpretation:**
* High alpha (0.8): Responsive, less smoothing
* Low alpha (0.2): More smoothing, less responsive

</details>

In [ ]:
# GRADED FUNCTION: exponential_smoothing_scratch
def exponential_smoothing_scratch(series, alpha=0.3):
    """
    Compute single exponential smoothing from scratch.
    
    Args:
        series: Time series data (array-like)
        alpha: Smoothing parameter (0 to 1)
    
    Returns:
        Smoothed values (numpy array, same length as input)
    """
    series = np.array(series)
    n = len(series)
    smoothed = np.zeros(n)
    
    ### START CODE HERE ### (≈ 6 lines)
    # Initialize with first observation
    smoothed[0] = None
    
    # Apply exponential smoothing recursively
    for t in range(1, n):
        # S_t = alpha * Y_t + (1 - alpha) * S_{t-1}
        smoothed[t] = None
    ### END CODE HERE ###
    
    return smoothed


In [ ]:
unittests.exercise_13(exponential_smoothing_scratch)

#### `forecast_with_exponential_smoothing_scratch` - Multi-Step Forecast

For simple exponential smoothing, the optimal multi-step forecast is the last smoothed value repeated for all future steps. Implement this using `exponential_smoothing_scratch`.

In [ ]:
# GRADED FUNCTION: forecast_with_exponential_smoothing_scratch
# GRADED FUNCTION: forecast_with_exponential_smoothing_scratch
def forecast_with_exponential_smoothing_scratch(series, alpha=0.3, steps=10):
    """
    Forecast future values using exponential smoothing from scratch.
    
    Args:
        series: Time series data (array-like)
        alpha: Smoothing parameter (0 to 1)
        steps: Number of steps to forecast
    
    Returns:
        Forecasts (numpy array of length 'steps')
    """
    ### START CODE HERE ### (≈ 4 lines)
    # Get smoothed values
    smoothed = None
    
    # For simple exponential smoothing, forecast is the last smoothed value
    forecast = None  # Repeat last value for all forecast steps
    ### END CODE HERE ###
    
    return forecast

In [ ]:
unittests.exercise_14(forecast_with_exponential_smoothing_scratch)

In [ ]:
# Test exponential smoothing from scratch with different alpha values
alphas = [0.1, 0.3, 0.5, 0.9]

plt.figure(figsize=(14, 8))
plt.plot(ts_data.index, ts_data.values, label='Original', 
         color='blue', alpha=0.3, linewidth=1)

colors = ['red', 'green', 'orange', 'purple']
for alpha, color in zip(alphas, colors):
    smoothed = exponential_smoothing_scratch(ts_data.values, alpha=alpha)
    plt.plot(ts_data.index, smoothed, label=f'ES (α={alpha})', 
             color=color, linewidth=2)

plt.title('Exponential Smoothing from Scratch - Different α Values', 
          fontsize=14, fontweight='bold')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compare with statsmodels
alpha_test = 0.3
scratch_smoothed = exponential_smoothing_scratch(train.values, alpha=alpha_test)
scratch_forecast = forecast_with_exponential_smoothing_scratch(
    train.values, alpha=alpha_test, steps=len(test)
)

# Statsmodels version
model = SimpleExpSmoothing(train)
fit = model.fit(smoothing_level=alpha_test, optimized=False)
statsmodels_forecast = fit.forecast(len(test))

# Compare forecasts
print(f"\nForecast Comparison (α={alpha_test}):")
print(f"Scratch forecast (first 5): {scratch_forecast[:5]}")
print(f"Statsmodels forecast (first 5): {statsmodels_forecast.values[:5]}")
print(f"\nMax difference: {np.max(np.abs(scratch_forecast - statsmodels_forecast.values)):.10f}")

# Calculate metrics
test_array = test.values
scratch_metrics = calculate_metrics(test_array, scratch_forecast)
statsmodels_metrics = calculate_metrics(test_array, statsmodels_forecast.values)

print(f"\nFrom Scratch Metrics:")
for metric, value in scratch_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nStatsmodels Metrics:")
for metric, value in statsmodels_metrics.items():
    print(f"  {metric}: {value:.4f}")

##### **Expected Output**
```
Forecast Comparison (α=0.3):
Scratch forecast (first 5): [XXX.XX XXX.XX XXX.XX XXX.XX XXX.XX]
Statsmodels forecast (first 5): [XXX.XX XXX.XX XXX.XX XXX.XX XXX.XX]

Max difference: 0.0000000000 (or very small)

From Scratch Metrics:
  MAE: XX.XXXX
  MSE: XXX.XXXX
  RMSE: XX.XXXX
  MAPE: XX.XXXX

Statsmodels Metrics:
  MAE: XX.XXXX
  MSE: XXX.XXXX
  RMSE: XX.XXXX
  MAPE: XX.XXXX
```
Higher α values should follow the data more closely (less smooth). Metrics should match statsmodels.

<a name='9'></a>
## 9 - Conclusion

**Congratulations!** You've completed the Time Series Analysis assignment!

### What You've Learned:

1. **Time Series Exploration**: Loaded data, visualized patterns, computed rolling statistics
2. **Decomposition**: Separated series into trend, seasonal, and residual components
3. **Stationarity Testing**: Applied ADF test and made series stationary through differencing
4. **Autocorrelation Analysis**: Computed ACF/PACF and identified model orders
5. **Simple Forecasting**: Implemented moving average and exponential smoothing
6. **Temporal Validation**: Created proper train/test splits and cross-validation
7. **Moving Average from Scratch**: Built simple and weighted moving averages
8. **Exponential Smoothing from Scratch**: Implemented single exponential smoothing

### Key Takeaways:

**Time Series Fundamentals:**
- Always respect temporal ordering
- Check for stationarity before modeling
- Understand trend, seasonality, and residuals
- Use proper validation techniques

**Model Selection Guide:**
- **Moving Average**: Simple baseline, lags behind trend
- **Exponential Smoothing**: Better for trending data, adaptable
- **ARIMA**: More sophisticated, handles autocorrelation
- **Seasonal Models**: For data with clear seasonality

**Best Practices:**
- Always visualize your data first
- Test for stationarity before modeling
- Use domain knowledge to set parameters
- Validate on out-of-sample data
- Check residuals for patterns
- Compare multiple forecasting methods
- Never shuffle time series data
- Don't extrapolate too far into future
- Don't ignore seasonality

### Common Pitfalls:

1. **Data Leakage**: Using future information to predict past
2. **Ignoring Stationarity**: Applying methods that assume stationarity to non-stationary data
3. **Over-differencing**: Making data too stationary, losing information
4. **Wrong Validation**: Using random splits instead of temporal
5. **Ignoring Seasonality**: Not accounting for regular patterns

### Real-World Applications:

**Finance:**
- Stock price prediction
- Risk management
- Portfolio optimization
- Algorithmic trading

**Business:**
- Sales forecasting
- Demand planning
- Inventory management
- Revenue prediction

**Operations:**
- Resource allocation
- Capacity planning
- Maintenance scheduling
- Supply chain optimization

**Science:**
- Climate modeling
- Epidemiology
- Sensor data analysis
- Signal processing

### Next Steps:

1. **Learn Advanced Models**: ARIMA, SARIMA, SARIMAX
2. **Explore Modern Methods**: Prophet, LSTM, Transformer models
3. **Study Seasonality**: Seasonal decomposition, Fourier features
4. **Master Multivariate**: VAR, VECM models
5. **Apply to Real Data**: Practice on Kaggle time series competitions
6. **Learn State Space Models**: Kalman filters, Dynamic Linear Models
7. **Study Probabilistic Forecasting**: Prediction intervals, quantile regression

### Recommended Resources:

- **Books**: "Forecasting: Principles and Practice" by Hyndman & Athanasopoulos
- **Courses**: Time series specialization on Coursera
- **Tools**: statsmodels, prophet, pmdarima, sktime
- **Practice**: Kaggle time series competitions

**Excellent work! You're now equipped to analyze and forecast temporal data! **